# Learning LangChain

# Setup

In [3]:
!pip install langchain
!pip install langchain-ollama
!pip install langchain-community
!pip install langchain-postgres

!pip install langchain-text-splitters
!pip install langchain-mcp-adapters

!pip install langgraph

# Env

In [ ]:
MODEL_DEEPSEEK_R1 = 'deepseek-r1'
MODEL_DEEPSEEK_R1_1_5B = 'deepseek-r1:1.5b'
MODEL_QWEN_3_5_0_8B = 'qwen3.5:0.8b' # tooling

# 1. LLM Fundamentals with LangChain

In [ ]:
# LLM
from langchain_ollama import OllamaLLM

# Key init args — generation params:
#     model: str/模型
#         Name of the Ollama model to use (e.g. `'llama4'`).
#     temperature: float | None/采样温度
#         Sampling temperature. Higher values make output more creative.
#     num_predict: int | None/预测token最大数量
#         Maximum number of tokens to predict.
#     top_k: int | None/下一个token的最大可能token数量
#         Limits the next token selection to the K most probable tokens.
#     top_p: float | None
#         Nucleus sampling parameter. Higher values lead to more diverse text.
#     mirostat: int | None
#         Enable Mirostat sampling for controlling perplexity.
#     seed: int | None
#         Random number seed for generation reproducibility.

# Key init args — client params:
#     base_url:
#         Base URL where Ollama server is hosted.
#     keep_alive:
#         How long the model stays loaded into memory.
#     format:
#         Specify the format of the output.

model = OllamaLLM(
    model=MODEL_DEEPSEEK_R1_1_5B,
    reasoning=False,
    temperature=0.7,
    num_predict=256,
)

res = model.invoke("Who are you?")
print(res)

I am DeepSeek's intelligent assistant, designed to provide thoughtful, helpful, and accurate responses to your questions. Whether you're looking for information, help with learning, or just want to chat, I'm here to assist you!


In [7]:
# Chat model

from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ChatMessage

# Key init args — completion params:
#     model: str
#         Name of Ollama model to use.
#     reasoning: bool | None
#         Controls the reasoning/thinking mode for
#         [supported models](https://ollama.com/search?c=thinking).

#         - `True`: Enables reasoning mode. The model's reasoning process will be
#             captured and returned separately in the `additional_kwargs` of the
#             response message, under `reasoning_content`. The main response
#             content will not include the reasoning tags.
#         - `False`: Disables reasoning mode. The model will not perform any reasoning,
#             and the response will not include any reasoning content.
#         - `None` (Default): The model will use its default reasoning behavior. Note
#             however, if the model's default behavior *is* to perform reasoning, think tags
#             (`<think>` and `</think>`) will be present within the main response content
#             unless you set `reasoning` to `True`.
#     temperature: float
#         Sampling temperature. Ranges from `0.0` to `1.0`.
#     num_predict: int | None
#         Max number of tokens to generate.
model = ChatOllama(
    model=MODEL_DEEPSEEK_R1,
    reasoning=False,
    temperature=0.7,
    num_predict=256,
)

# prompt = [HumanMessage('What is the captial of France?')]
# model.invoke(prompt)

system_msg = SystemMessage('You are a helpful assistant that responds to questions with three exclamation marks.')
human_msg = HumanMessage('What is the capital of France?')
model.invoke([system_msg, human_msg])

AIMessage(content='Paris!!!', additional_kwargs={}, response_metadata={'model': 'deepseek-r1', 'created_at': '2026-03-18T01:19:34.7369684Z', 'done': True, 'done_reason': 'stop', 'total_duration': 4792520100, 'load_duration': 2155154600, 'prompt_eval_count': 28, 'prompt_eval_duration': 2276104000, 'eval_count': 3, 'eval_duration': 319834900, 'logprobs': None, 'model_name': 'deepseek-r1', 'model_provider': 'ollama'}, id='lc_run--019cfe86-c096-74a0-8c36-2bc47e7e2418-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 28, 'output_tokens': 3, 'total_tokens': 31})

In [11]:
# Prompt template
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

template = PromptTemplate.from_template("""Answer the question based on the
context below. If the question cannot be answered using the information
provided, answer with "I don't know".
Context: {context}
Question: {question}
Answer: """)

prompt = template.invoke({
    "context": """The most recent advancements in NLP are being driven by Large
Language Models (LLMs). These models outperform their smaller
counterparts and have become invaluable for developers who are creating
applications with NLP capabilities. Developers can tap into these
models through Hugging Face's `transformers` library, or by utilizing
OpenAI and Cohere's offerings through the `openai` and `cohere`
libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
})

model = OllamaLLM(
    model=MODEL_DEEPSEEK_R1,
    reasoning=False,
    temperature=0.7,
    num_predict=256,
)
print(model.invoke(prompt))

chat_template = ChatPromptTemplate.from_messages([
    ('system', '''Answer the question based on the context below. If the question cannot be answered using the information provided, answer with "I don\'t know".'''),
    ('human', 'Context: {context}'),
    ('human', 'Question: {question}'),
])
chat_prompt = chat_template.invoke({
    "context": """The most recent advancements in NLP are being driven by
Large Language Models (LLMs). These models outperform their smaller
counterparts and have become invaluable for developers who are creating
applications with NLP capabilities. Developers can tap into these 
models through Hugging Face's `transformers` library, or by utilizing
OpenAI and Cohere's offerings through the `openai` and `cohere`
libraries, respectively.""",
    "question": "Which model providers offer LLMs?"
})

model = ChatOllama(
    model=MODEL_DEEPSEEK_R1,
    reasoning=False,
    temperature=0.7,
    num_predict=256,
)
print(model.invoke(chat_prompt))

Hugging Face, OpenAI, and Cohere offer Large Language Models (LLMs).
content='The model providers mentioned in the context are:\n\n- Hugging Face (via their `transformers` library)\n- OpenAI (via their `openai` library)\n- Cohere (via their `cohere` library)\n\nThese providers offer Large Language Models (LLMs) that developers can utilize. \n\nNote: The context does not specify which specific models are available from these providers, only that they offer LLMs.' additional_kwargs={} response_metadata={'model': 'deepseek-r1', 'created_at': '2026-03-18T01:27:01.3969969Z', 'done': True, 'done_reason': 'stop', 'total_duration': 23505830200, 'load_duration': 167092700, 'prompt_eval_count': 134, 'prompt_eval_duration': 10224611900, 'eval_count': 87, 'eval_duration': 13068658800, 'logprobs': None, 'model_name': 'deepseek-r1', 'model_provider': 'ollama'} id='lc_run--019cfe8d-4841-73c3-9f89-e66bdb764730-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 134, 'output_tokens':

In [ ]:
# JSON output
from langchain_ollama import ChatOllama
from pydantic import BaseModel

class AnswerWithJustification(BaseModel):
  '''An answer to the user's question along with justification for the answer.'''
  answer: str
  '''The answer to the user's question'''
  justification: str
  '''Justification for the answer'''

llm = ChatOllama(model=MODEL_DEEPSEEK_R1_1_5B, temperature=0, reasoning=False)
structured_llm = llm.with_structured_output(AnswerWithJustification)
structured_llm.invoke('What weighs more, a pound of bricks or a pound of feathers')

AnswerWithJustification(answer='A pound of bricks weighs more than a pound of feathers because bricks are made of dense materials with higher density, while feathers are lightweight and spread out in size. Even though both weigh one pound, the bricks occupy more volume due to their denser composition.', justification='The weight of an object is determined by its mass multiplied by gravity. Both a pound of bricks and a pound of feathers have the same weight because they are measured using the same unit (pound). However, density plays a role in how much volume each object occupies. Bricks, being made of dense materials like clay or stone, have higher density and thus occupy more space for the same mass. Feathers, on the other hand, are made of less dense materials and are smaller in size, resulting in more volume per pound. Therefore, a pound of bricks weighs more than a pound of feathers.')

In [14]:
# Output parsers
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()
parser.invoke('apple, banana, cheery')

['apple', 'banana', 'cheery']

In [ ]:
# Assemble

# LECL: LangChain Expression Language
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant.'),
    ('human', '{question}'),
])

model = ChatOllama(model=MODEL_DEEPSEEK_R1, temperature=0, reasoning=False)
chatbot = template | model
# sync
# alternative: stream, async
chatbot.invoke({"question": "Which model providers offer LLMs?"})

AIMessage(content="There are several prominent model providers that offer Large Language Models (LLMs). Here are some of the most well-known ones:\n\n1. **OpenAI**: Provides models like GPT-3, GPT-4, and Codex. It offers APIs for developers to integrate these models into their applications.\n\n2. **Google**: Offers its own suite of AI tools, including the Gemini model (previously Bard) and also provides access to its language models through Google Cloud AI and APIs.\n\n3. **Anthropic**: Known for its Claude series of models, which are designed to be helpful, harmless, and honest.\n\n4. **Meta**: Develops models like LLaMA (Large Language Model Meta AI), which is available for research and development purposes.\n\n5. **Mistral**: Provides open-source models like Mistral-7B and Mistral-13B, which are popular in the community.\n\n6. **Cohere**: Offers models like Command and Command Light, designed for tasks like content generation, summarization, and question answering.\n\n7. **ElevenLab

# 2. RAG Part I: Indexing Your Data

In [ ]:
from langchain_community.document_loaders import TextLoader


In [ ]:
!pip install beautifulsoup4

from langchain_community.document_loaders import WebBaseLoader

In [ ]:
!pip install pypdf
from langchain_community.document_loaders import PyPDFLoader


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import Language

In [ ]:
#  https://reference.langchain.com/python/langchain-ollama/embeddings/OllamaEmbeddings
from langchain_ollama import OllamaEmbeddings

embed = OllamaEmbeddings(model="llama3")

In [ ]:
# https://github.com/pgvector/pgvector
from langchain_postgres.vectorstores import PGVector
from langchain_core.documents import Document


from langchain.indexes import SQLRecordManager, index
from langchain.docstore.document import Document

In [ ]:
# indexing optimization
from langchain.retrievers.multi_vector import MultiVectorRetriever

In [ ]:
# RAPTOR: Recursive Abstrative Processing for Tree-Organized Retrieval

In [ ]:
# ColBERT
!pip install ragatouille

from ragatouille import RAGPretrainedModel
RAG = RAGPretrainedModel.from_pretrained("colbert-ir/colbertv2.0")

# 3. RAG Part II: Chatting with Your Data

In [ ]:
from langchain.retrievers.self_query.base import SelfQueryRetriever

In [ ]:
from langchain.chains import create_sql_query_chain

# 4. Using LangGraph to Add Memory to Your Chatbot

# 5. Cognitive Architectures with LangGraph

In [4]:
from langgraph.graph import StateGraph

In [5]:
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
from langchain_core.messages import trim_messages, filter_messages, merge_message_runs

# 6. Agent Architecture

In [7]:
!pip install duckduckgo-search

# 7. Agents II

# 8. Patterns to Make the Most of LLMs

# 9. Deployment: Launching Your AI Application into Production

In [20]:
!pip install "langgraph-cli[inmem]"

  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
Using cached setuptools-82.0.1-py3-none-any.whl (1.0 MB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.6.0
    Uninstalling setuptools-75.6.0:
      Successfully uninstalled setuptools-75.6.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda 25.1.1 requires conda-libmamba-solver>=24.11.0, but you have conda-libmamba-solver 24.9.0 which is incompatible.
pymilvus 2.5.14 requires grpcio<=1.67.1,>=1.49.1, but you have grpcio 1.78.0 which is incompatible.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.3 which is incompatible.
tensorflow-intel 2.18.0 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.3, but you have protobuf 6.33.5 which is incompatible.
tensorflow-intel 2.18.0 requires tensorboard<2.19,>=2.18, but you have tensorboard 2.20.0 which is incompatible.


In [13]:
!bash langchain_runs.sh

event: metadata
data: {"run_id":"019d04a4-afd8-7f70-997d-be5b83a4b393","attempt":1}

event: error
data: {"error":"UnboundLocalError","message":"An internal error occurred"}



In [31]:
# Client SDK
from langgraph_sdk import get_client
# only pass the url argument to get_client() if you changed the default port
# when calling langgraph up
client = get_client(url='http://127.0.0.1:2024')
# Using the graph deployed with the name "agent"
assistant_id = "agent"
thread = await client.threads.create()
# 如何调试失败???
input = {"messages": [
    {
        "role": "user",
        "content": "1 + 1 = ?"
        # "content": "What's the time now?"
    }
]}
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id,
    input=input,
    stream_mode="updates",
):
  print(f"Receiving new event of type: {chunk.event}...")
  print(chunk.data)
  print("\n\n")

Receiving new event of type: metadata...
{'run_id': '019d04cb-0c9e-7eb1-a5b4-c3af6c682c31', 'attempt': 1}



Receiving new event of type: updates...
{'model': {'messages': [{'content': '', 'additional_kwargs': {}, 'response_metadata': {'model': 'qwen3.5:0.8b', 'created_at': '2026-03-19T06:31:55.8024927Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5897850900, 'load_duration': 223090400, 'prompt_eval_count': 355, 'prompt_eval_duration': 1873832100, 'eval_count': 87, 'eval_duration': 3723878600, 'logprobs': None, 'model_name': 'qwen3.5:0.8b', 'model_provider': 'ollama'}, 'type': 'ai', 'name': 'simple_agent', 'id': 'lc_run--019d04cb-0f8e-7fa0-b0ea-0b9670807bdf-0', 'tool_calls': [{'name': 'calculator', 'args': {'expression': '1 + 1'}, 'id': 'f9638553-2b73-4af6-9cb1-b6d13e78cbe1', 'type': 'tool_call'}], 'invalid_tool_calls': [], 'usage_metadata': {'input_tokens': 355, 'output_tokens': 87, 'total_tokens': 442}}]}}



Receiving new event of type: updates...
{'tools': {'messages': 

# 10. Testing: Evaluation, Monitoring, and Continuous Improvement

# 11. Building with LLMs

# Cleanup

In [1]:
!pip uninstall -y langchain
!pip uninstall -y langchain-ollama
!pip uninstall -y langchain-community
!pip uninstall -y langchain-postgres

!pip uninstall -y langchain-text-splitters
!pip uninstall -y langchain-mcp-adapters

Found existing installation: langchain 0.3.25
Uninstalling langchain-0.3.25:
  Successfully uninstalled langchain-0.3.25
Found existing installation: langchain-ollama 0.3.3
Uninstalling langchain-ollama-0.3.3:
  Successfully uninstalled langchain-ollama-0.3.3
Found existing installation: langchain-community 0.3.24
Uninstalling langchain-community-0.3.24:
  Successfully uninstalled langchain-community-0.3.24
Found existing installation: langchain-postgres 0.0.17
Uninstalling langchain-postgres-0.0.17:
  Successfully uninstalled langchain-postgres-0.0.17
Found existing installation: langchain-text-splitters 0.3.8
Uninstalling langchain-text-splitters-0.3.8:
  Successfully uninstalled langchain-text-splitters-0.3.8
Found existing installation: langchain-mcp-adapters 0.2.2
Uninstalling langchain-mcp-adapters-0.2.2:
  Successfully uninstalled langchain-mcp-adapters-0.2.2


In [ ]:
!pip uninstall -y duckduckgo-search

In [ ]:
!pip uninstall -y "langgraph-cli[inmem]"